In [1]:
!pip install -q -U transformers accelerate requests

import os
import json
import glob
import time
import zlib
import numpy as np
import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.3 MB/s eta 0:00:00
device: cuda


In [2]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

layers = model.model.language_model.layers
n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = hidden_dim // n_head
IMAGE_TOKEN_ID = getattr(model.config, "image_token_id", None) or model.config.image_token_index
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim, image token id", IMAGE_TOKEN_ID)

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

32 language model layers, 32 heads, 128 head dim, image token id 32000


In [3]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
img_dir = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]

YES_ID, NO_ID = 3869, 1939
NOISE_LEVELS = [1.5, 1.25, 1.0]
GRAD_SCALES = [32.0, 1.0]
GRAD_SCALE = GRAD_SCALES[0]
TAIL_N = 4
GROUPS = ["visual", "text", "tail"]
BATCH_SAVE_SIZE = 200
OUT_DIR = "/kaggle/working/atp_llava"
os.makedirs(OUT_DIR, exist_ok=True)


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id):
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    if qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
assert [processor.tokenizer.decode([i]).strip() for i in (YES_ID, NO_ID)] == ["Yes", "No"], processor.tokenizer.convert_ids_to_tokens([YES_ID, NO_ID])

with open(f"{INPUT}/atp-llava-selection/selection.json") as f:
    selection = json.load(f)
all_sel = [qid for t in selection["selected"] for qid in selection["selected"][t]]
assert len(all_sel) == len(set(all_sel)) == 8403, "attach atp-llava-selection version 1"
print(len(all_sel), "selected questions")


def prep(qid):
    q = question_by_id[qid]
    inputs = build_inputs(load_image(resolve_image_path(q)), q["question_text"])
    ids = inputs["input_ids"]
    T = ids.shape[1]
    vis = ids[0] == IMAGE_TOKEN_ID
    tail = torch.zeros(T, dtype=torch.bool, device=ids.device)
    tail[-TAIL_N:] = True
    text = ~vis & ~tail
    assert int(vis.sum()) == 576 and not bool((vis & tail).any())
    return {
        "qid": qid, "T": T, "input_ids": ids, "pixel_values": inputs["pixel_values"],
        "attention_mask": inputs["attention_mask"], "vis": vis,
        "masks": {"visual": vis.float().cpu(), "text": text.float().cpu(), "tail": tail.float().cpu()},
        "sign": 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0,
    }

8403 selected questions


In [4]:
STATE = {"noise": None, "seed": 0, "capture": False, "grad": False, "live": None, "vis": None}


def _layer_pre(l):
    def hook(module, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        changed = False
        if l == 0:
            if STATE["noise"] is not None:
                vis = STATE["vis"]
                h = h.clone()
                x = h[:, vis].float()
                generator = torch.Generator().manual_seed(STATE["seed"])
                noise = torch.randn(x.shape, generator=generator).to(h.device)
                h[:, vis] = (x + noise * x.std() * STATE["noise"]).to(h.dtype)
                changed = True
            if STATE["grad"]:
                h = h.detach().requires_grad_(True)
                changed = True
        if STATE["capture"]:
            STATE["live"]["rp"][l] = h
        if not changed:
            return None
        if args:
            return (h,) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs)
        kwargs["hidden_states"] = h
        return args, kwargs
    return hook


def _z_pre(l):
    def hook(module, args):
        if STATE["capture"]:
            STATE["live"]["z"][l] = args[0]
    return hook


def _mlp_fwd(l):
    def hook(module, args, output):
        if STATE["capture"]:
            STATE["live"]["mlp"][l] = output
    return hook


_handles = []
for l, layer in enumerate(layers):
    _handles.append(layer.register_forward_pre_hook(_layer_pre(l), with_kwargs=True))
    _handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_z_pre(l)))
    _handles.append(layer.mlp.register_forward_hook(_mlp_fwd(l)))
print(len(_handles), "hooks registered")

96 hooks registered


In [5]:
def mask_on(p, gname, dev):
    cache = p.setdefault("_mask", {})
    key = (gname, str(dev))
    if key not in cache:
        cache[key] = p["masks"][gname].to(dev)
    return cache[key]


def capture_run(p, noise, seed):
    STATE.update(noise=noise, seed=seed, capture=True, grad=True, vis=p["vis"])
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    with torch.enable_grad():
        out = model(
            input_ids=p["input_ids"], pixel_values=p["pixel_values"], attention_mask=p["attention_mask"], use_cache=False,
        )
        logits = out.logits[0, -1].float()
        delta = p["sign"] * (logits[YES_ID] - logits[NO_ID])
        live = STATE["live"]
        grads = torch.autograd.grad(delta * GRAD_SCALE, live["z"] + live["mlp"] + live["rp"])
    finite = all(bool(torch.isfinite(g).all()) for g in grads)
    acts = {k: [t.detach() for t in live[k]] for k in ("z", "mlp", "rp")}
    G = {"z": list(grads[:n_layer]), "mlp": list(grads[n_layer:2 * n_layer]), "rp": list(grads[2 * n_layer:])}
    result = {"acts": acts, "grads": G, "delta": float(delta.detach()), "finite": finite}
    STATE.update(capture=False, grad=False, live=None, noise=None)
    del out, logits, delta, live, grads
    return result


@torch.no_grad()
def capture_forward(p, noise, seed):
    STATE.update(noise=noise, seed=seed, capture=True, grad=False, vis=p["vis"])
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    out = model(
        input_ids=p["input_ids"], pixel_values=p["pixel_values"], attention_mask=p["attention_mask"], use_cache=False,
    )
    logits = out.logits[0, -1].float()
    delta = p["sign"] * (logits[YES_ID] - logits[NO_ID])
    result = {"acts": STATE["live"], "delta": float(delta)}
    STATE.update(capture=False, live=None, noise=None)
    del out, logits
    return result


def atp_effects(A, B, G, p):
    head = torch.zeros(len(GROUPS), n_layer, n_head)
    mlp = torch.zeros(len(GROUPS), n_layer)
    rp = torch.zeros(len(GROUPS), n_layer)
    T = p["T"]
    for l in range(n_layer):
        dz = ((A["z"][l].float() - B["z"][l].float()) * G["z"][l].float() / GRAD_SCALE)[0].view(T, n_head, head_dim).sum(-1)
        dm = ((A["mlp"][l].float() - B["mlp"][l].float()) * G["mlp"][l].float() / GRAD_SCALE)[0].sum(-1)
        dr = ((A["rp"][l].float() - B["rp"][l].float()) * G["rp"][l].float() / GRAD_SCALE)[0].sum(-1)
        for gi, g in enumerate(GROUPS):
            head[gi, l] = (mask_on(p, g, dz.device) @ dz).cpu()
            mlp[gi, l] = (mask_on(p, g, dm.device) @ dm).cpu()
            rp[gi, l] = (mask_on(p, g, dr.device) @ dr).cpu()
    return {"head": head, "mlp": mlp, "rp": rp}


def process(p, seed):
    global GRAD_SCALE
    clean, scale_used = None, None
    for scale in GRAD_SCALES:
        GRAD_SCALE = scale
        c = capture_run(p, None, seed)
        if c["finite"] and np.isfinite(c["delta"]):
            clean, scale_used = c, scale
            break
        del c
    if clean is None:
        GRAD_SCALE = GRAD_SCALES[0]
        return None, "clean_nonfinite"
    result, reason = None, "corrupt_nonfinite"
    for noise in NOISE_LEVELS:
        r = capture_forward(p, noise, seed)
        if np.isfinite(r["delta"]):
            eff = atp_effects(r["acts"], clean["acts"], clean["grads"], p)
            if all(bool(torch.isfinite(eff[k]).all()) for k in eff):
                result = {
                    "delta_clean": clean["delta"], "delta_corrupt": r["delta"], "TE": clean["delta"] - r["delta"],
                    "noise_used": noise, "grad_scale_used": scale_used,
                    "head": eff["head"], "mlp": eff["mlp"], "rp": eff["rp"],
                }
                del r
                break
        del r
    GRAD_SCALE = GRAD_SCALES[0]
    del clean
    return result, reason

In [6]:
def collect(tag, keep):
    todo_ids = [qid for qid in all_sel if keep(qid)]
    paths = sorted(glob.glob(os.path.join(OUT_DIR, f"{tag}_batch_*.pt")))
    done_ids = set()
    for path in paths:
        done_ids.update(e["question_id"] for e in torch.load(path))
    todo = [qid for qid in todo_ids if qid not in done_ids]
    print(f"{tag}: {len(todo_ids)} questions, {len(done_ids)} already done, {len(todo)} to run")

    batch_entries, batch_index, missing, failed = [], len(paths), 0, 0
    t_start = time.time()

    for i, qid in enumerate(todo):
        q = question_by_id[qid]
        image_path = resolve_image_path(q)
        if image_path is None or not os.path.exists(image_path):
            missing += 1
            continue

        p = prep(qid)
        seed = zlib.crc32(qid.encode()) % (2 ** 31)
        result, reason = process(p, seed)
        entry = {
            "question_id": qid, "image_id": q["image_id"], "hallucination_type": q["hallucination_type"],
            "ground_truth_answer": q["ground_truth_answer"], "stratum": selection["stratum"][qid],
            "group": selection["group"][qid], "source": selection["source"][qid], "split": selection["split"][qid],
            "sign": p["sign"], "seed": seed, "ok": result is not None,
        }
        if result is None:
            entry["reason"] = reason
            failed += 1
        else:
            entry.update(result)
            entry["pred_yes"] = bool(p["sign"] * result["delta_clean"] > 0)
        batch_entries.append(entry)
        del p

        if len(batch_entries) >= BATCH_SAVE_SIZE:
            torch.save(batch_entries, os.path.join(OUT_DIR, f"{tag}_batch_{batch_index}.pt"))
            batch_entries, batch_index = [], batch_index + 1

        if i % 100 == 0 and i > 0:
            torch.cuda.empty_cache()
            elapsed = time.time() - t_start
            avg = elapsed / (i + 1)
            print(f"[{i + 1}/{len(todo)}] {qid} ({avg:.2f} s/question avg, ETA {avg * (len(todo) - i - 1) / 60:.1f} min, "
                  f"failed {failed}, missing images {missing})")

    if batch_entries:
        torch.save(batch_entries, os.path.join(OUT_DIR, f"{tag}_batch_{batch_index}.pt"))
    print(f"done -- {tag}, failed {failed}, missing images {missing}, {(time.time() - t_start) / 60:.1f} min")
    return missing

In [7]:
missing = collect("main", lambda qid: not qid.startswith("reefknot_"))
assert missing == 0, f"{missing} questions have no image"

main: 5582 questions, 0 already done, 5582 to run
[101/5582] amber_8548 (1.60 s/question avg, ETA 146.4 min, failed 0, missing images 0)
[201/5582] amber_2532 (1.70 s/question avg, ETA 152.7 min, failed 0, missing images 0)
[301/5582] amber_1780 (1.72 s/question avg, ETA 151.5 min, failed 0, missing images 0)
[401/5582] amber_4212 (1.73 s/question avg, ETA 149.3 min, failed 0, missing images 0)
[501/5582] amber_6112 (1.74 s/question avg, ETA 147.0 min, failed 0, missing images 0)
[601/5582] amber_1306 (1.74 s/question avg, ETA 144.3 min, failed 0, missing images 0)
[701/5582] amber_5006 (1.74 s/question avg, ETA 141.5 min, failed 0, missing images 0)
[801/5582] amber_4153 (1.74 s/question avg, ETA 138.4 min, failed 0, missing images 0)
[901/5582] amber_4694 (1.74 s/question avg, ETA 135.6 min, failed 0, missing images 0)
[1001/5582] amber_3772 (1.74 s/question avg, ETA 132.8 min, failed 0, missing images 0)
[1101/5582] amber_3263 (1.74 s/question avg, ETA 130.0 min, failed 0, missing i

In [8]:
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
    if os.path.exists(zpath):
        os.remove(zpath)
print("Visual Genome downloaded, extracted, zips removed")

--2026-10-09 03:07:03--  https://cs.stanford.edu/people/rak248/VG_100K/images.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9730308001 (9.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K.zip’

/kaggle/working/VG_ 100%[===================>]   9.06G  12.0MB/s    in 30m 2s  

2026-10-09 03:37:05 (5.15 MB/s) - ‘/kaggle/working/VG_100K.zip’ saved [9730308001/9730308001]

--2026-10-09 03:37:06--  https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5471658058 (5.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K_2.zip’

/kaggle/working/VG_ 100%[===================>]   5.10G  14.3MB/s    in 6m 40s  

2026-10-09 03:43:46 (13.0 MB

In [9]:
missing = collect("reefknot", lambda qid: qid.startswith("reefknot_"))
assert missing == 0, f"{missing} questions have no image"

reefknot: 2821 questions, 0 already done, 2821 to run
[101/2821] reefknot_2338608_0 (1.71 s/question avg, ETA 77.5 min, failed 0, missing images 0)
[201/2821] reefknot_2336272_1 (1.70 s/question avg, ETA 74.3 min, failed 0, missing images 0)
[301/2821] reefknot_2319112_1 (1.70 s/question avg, ETA 71.3 min, failed 0, missing images 0)
[401/2821] reefknot_2353350_0 (1.70 s/question avg, ETA 68.4 min, failed 0, missing images 0)
[501/2821] reefknot_2412845_1 (1.70 s/question avg, ETA 65.6 min, failed 0, missing images 0)
[601/2821] reefknot_2399956_1 (1.70 s/question avg, ETA 62.7 min, failed 0, missing images 0)
[701/2821] reefknot_1592236_0 (1.70 s/question avg, ETA 59.9 min, failed 0, missing images 0)
[801/2821] reefknot_2358567_1 (1.70 s/question avg, ETA 57.1 min, failed 0, missing images 0)
[901/2821] reefknot_2373131_1 (1.69 s/question avg, ETA 54.2 min, failed 0, missing images 0)
[1001/2821] reefknot_2414749_0 (1.69 s/question avg, ETA 51.4 min, failed 0, missing images 0)
[1101

In [10]:
entries = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, "*_batch_*.pt"))):
    entries.extend(torch.load(path))
ids = [e["question_id"] for e in entries]
print(f"{len(entries)} entries, {len(set(ids))} unique, {len(set(all_sel) - set(ids))} selected questions missing")
assert set(ids) == set(all_sel) and len(ids) == len(set(ids)), "collection incomplete or duplicated"

good = [e for e in entries if e["ok"]]
bad = [e for e in entries if not e["ok"]]
print(f"ok {len(good)}, failed {len(bad)}", {r: sum(e['reason'] == r for e in bad) for r in {e['reason'] for e in bad}})
print("noise level used:", {n: sum(e["noise_used"] == n for e in good) for n in NOISE_LEVELS})
print("gradient scale used:", {s: sum(e["grad_scale_used"] == s for e in good) for s in GRAD_SCALES})

dla_ld = {}
for path in glob.glob(f"{INPUT}/dla-llava/*_batch_*.pt"):
    for e in torch.load(path):
        if e["question_id"] in set(ids):
            dla_ld[e["question_id"]] = float(e["logit_diff"])
err = max(abs(e["sign"] * e["delta_clean"] - dla_ld[e["question_id"]]) for e in good)
print(f"clean logit difference vs dla-llava over {len(good)} questions: max err {err:.4f}")
assert err < 0.1, "clean runs differ from the DLA collection"
assert len(bad) <= 0.02 * len(entries), "too many failed questions"

for stratum in ("false_yes", "true_yes", "true_no"):
    sub = [e for e in good if e["stratum"] == stratum]
    te = np.array([e["TE"] for e in sub])
    print(f"{stratum:10s} n {len(sub):5d} | mean TE {te.mean():+.2f} | share with |TE| > 1: {np.mean(np.abs(te) > 1):.2f} | mean delta clean {np.mean([e['delta_clean'] for e in sub]):+.2f}")
print("collection verified")

8403 entries, 8403 unique, 0 selected questions missing
ok 8403, failed 0 {}
noise level used: {1.5: 8403, 1.25: 0, 1.0: 0}
gradient scale used: {32.0: 8403, 1.0: 0}
clean logit difference vs dla-llava over 8403 questions: max err 0.0000
false_yes  n  3358 | mean TE -0.67 | share with |TE| > 1: 0.32 | mean delta clean -1.29
true_yes   n  3028 | mean TE +1.52 | share with |TE| > 1: 0.53 | mean delta clean +2.12
true_no    n  2017 | mean TE +1.82 | share with |TE| > 1: 0.76 | mean delta clean +1.30
collection verified


In [11]:
with open(os.path.join(OUT_DIR, "atp_meta.json"), "w") as f:
    json.dump({
        "model": MODEL_NAME, "yes_id": YES_ID, "no_id": NO_ID, "n_layer": n_layer, "n_head": n_head, "groups": GROUPS,
        "tail_n": TAIL_N, "noise_levels": NOISE_LEVELS, "grad_scales": GRAD_SCALES, "direction": "noising",
        "n_ok": len(good), "n_failed": len(bad),
    }, f)

with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "atp-llava-noising",
        "id": f"{USERNAME}/atp-llava-noising",
        "subtitle": "Noising attribution patching effects for LLaVA yes/no questions",
        "description": "Per-question attribution patching for LLaVA-1.5-7B on the 8403 questions in atp-llava-selection. "
                       "Corrupt run: Gaussian noise added to the 576 visual embeddings at layer 0 with std noise_used times the std of those embeddings. "
                       "Metric delta is the sign-corrected Yes minus No logit gap at the last position, positive when the model favours the true answer. "
                       "head [3, 32, 32], mlp [3, 32] and rp [3, 32] hold the first-order effect of replacing each component's clean activation with its corrupt activation, "
                       "(corrupt minus clean) times the gradient of delta at the clean run, in logit units. The first axis is the position group: visual, text, tail (last 4 positions). "
                       "head is the attention head output before o_proj, mlp the MLP output, rp the residual stream entering the layer. "
                       "TE is delta_clean minus delta_corrupt. Entries with ok false have no effects.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

!kaggle datasets create -p {OUT_DIR}

Starting upload for file main_batch_3.pt
100%|██████████████████████████████████████| 2.69M/2.69M [00:00<00:00, 10.6MB/s]
Upload successful: main_batch_3.pt (3MB)
Starting upload for file main_batch_6.pt
100%|██████████████████████████████████████| 2.69M/2.69M [00:00<00:00, 10.5MB/s]
Upload successful: main_batch_6.pt (3MB)
Starting upload for file main_batch_24.pt
100%|██████████████████████████████████████| 2.69M/2.69M [00:00<00:00, 11.6MB/s]
Upload successful: main_batch_24.pt (3MB)
Starting upload for file reefknot_batch_5.pt
100%|██████████████████████████████████████| 2.70M/2.70M [00:00<00:00, 11.2MB/s]
Upload successful: reefknot_batch_5.pt (3MB)
Starting upload for file reefknot_batch_10.pt
100%|██████████████████████████████████████| 2.70M/2.70M [00:00<00:00, 11.0MB/s]
Upload successful: reefknot_batch_10.pt (3MB)
Starting upload for file reefknot_batch_4.pt
100%|██████████████████████████████████████| 2.70M/2.70M [00:00<00:00, 11.6MB/s]
Upload successful: reefknot_batch_4.pt 